In [ ]:
import os
import glob
import sys
from datetime import datetime
import matplotlib.pyplot as plt
import random
import cv2
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import Union
from dataclasses import dataclass, asdict
import math

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
print(f'Number of available cpu: {os.cpu_count()}')

In [ ]:
!pip install albumentations
!pip install -q wandb torchmetrics
!pip install torchmetrics[image]

# Data Pipeline

In [ ]:
# Standard library imports
import os
import json
import time
import random
import math
from dataclasses import dataclass
from pathlib import Path
from collections import Counter
from typing import Dict, List, Tuple, Optional

# Third-party imports
import numpy as np
import torch
import albumentations as A
import cv2
from PIL import Image, ImageOps

# PyTorch data utilities
from torch.utils.data import Dataset, DataLoader

# Global constants...
TARGET_CATEGORIES = [
    "10_dress",
    "8_skirt",
    "43_ruffle",
    "1_top__t_shirt__sweatshirt",
    "0_shirt__blouse",
    "4_jacket",
    "9_coat",
    "2_sweater",
    "3_cardigan",
    "5_vest",
    "6_pants",
    "7_shorts",
]
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


### 1) Pipeline Configuration
Define tunable configuration for dataset paths, target categories, and sketch sampling ratios.

In [ ]:
# [Removed] PipelineConfig is no longer needed because the dataset is already preprocessed.


### 2) Image Preprocessing Helpers
Prepare utility functions for consistent padding, resizing, and category-wise file indexing.

In [ ]:
# [Removed] list_category_images is no longer needed because the dataset is already preprocessed.


### 3) Build Paired Metadata
Scan categories, match real and sketch files by stem, and assign sketch methods using the configured sampling ratios.

In [ ]:
# [Removed] build_gan_pairs is no longer needed because the dataset is already preprocessed.


### 4) Dataset and DataLoader
Define the PyTorch dataset and a reusable DataLoader builder with safe defaults.

In [ ]:
# Cell 5
import os
from typing import List, Tuple, Optional
import numpy as np
import torch
import albumentations as A
import cv2
from PIL import Image
from torch.utils.data import Dataset, DataLoader

class SketchToRealGANDataset(Dataset):
    """PyTorch dataset that loads paired sketch-real images."""

    def __init__(
        self,
        rows: List[dict],
        image_size: int = 256,
        apply_augmentation: bool = False,
        flip_prob: float = 0.5,
        crop_scale: Tuple[float, float] = (0.9, 1.0),
        max_translate_ratio: float = 0.08,
        max_rotation_deg: float = 10.0,
        scale_range: Tuple[float, float] = (0.95, 1.05),
        # Thêm các tham số cho việc làm đứt nét Sketch
        sketch_degrade_prob: float = 0.7,
        degrade_holes_range: Tuple[int, int] = (5, 20),
        degrade_size_range: Tuple[int, int] = (10, 40),
    ):
        self.rows = rows
        self.image_size = image_size
        self.apply_augmentation = apply_augmentation
        
        # --- 1. SPATIAL AUGMENTATION (Áp dụng chung cho cả Real và Sketch để giữ align) ---
        max_shift_percent = max_translate_ratio * 100.0
        self._spatial_augment = A.Compose(
            [
                A.HorizontalFlip(p=flip_prob),
            ],
            additional_targets={"real": "image"},
        )

        # --- 2. SKETCH DEGRADATION (Chỉ áp dụng cho Sketch để ép model học nội suy) ---
        self._sketch_only_augment = A.Compose([])

    def _load_img(self, path: str) -> np.ndarray:
        img_bgr = cv2.imread(path)
        if img_bgr is None:
            raise ValueError(f"Could not load image: {path}")
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        if img_rgb.shape[0] != self.image_size or img_rgb.shape[1] != self.image_size:
            img_rgb = cv2.resize(img_rgb, (self.image_size, self.image_size), interpolation=cv2.INTER_LINEAR)
        return img_rgb

    def _apply_spatial_augment_pair(
        self,
        sketch_img: np.ndarray,
        real_img: np.ndarray,
    ) -> Tuple[np.ndarray, np.ndarray]:
        transformed = self._spatial_augment(image=sketch_img, real=real_img)
        return transformed["image"], transformed["real"]

    def _to_tensor(self, img: np.ndarray) -> torch.Tensor:
        arr = img.astype(np.float32) / 127.5 - 1.0
        return torch.from_numpy(arr).permute(2, 0, 1)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        sketch_img = self._load_img(row["sketch_path"])
        real_img = self._load_img(row["real_path"])
        
        if self.apply_augmentation:
            # Bước 1: Augment không gian cho cả 2 ảnh để giữ sự đồng nhất
            sketch_img, real_img = self._apply_spatial_augment_pair(sketch_img, real_img)
            
            # Bước 2: Chỉ làm hỏng (degrade) ảnh Sketch
            sketch_img = self._sketch_only_augment(image=sketch_img)["image"]

        return {
            "sketch": self._to_tensor(sketch_img),
            "real": self._to_tensor(real_img),
            "filename_stem": row["filename_stem"],
        }


def build_gan_dataloader(
    rows: List[dict],
    batch_size: int = 16,
    image_size: int = 256,
    shuffle: bool = True,
    num_workers: Optional[int] = None,
    apply_augmentation: bool = False,
    sketch_degrade_prob: float = 0.7, 
) -> DataLoader:
    """Build a DataLoader from paired metadata rows."""
    if num_workers is None:
        num_workers = min(4, os.cpu_count() if os.cpu_count() is not None else 4)

    dataset = SketchToRealGANDataset(
        rows=rows,
        image_size=image_size,
        apply_augmentation=apply_augmentation,
        sketch_degrade_prob=sketch_degrade_prob,
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
        persistent_workers=True if num_workers > 0 else False,
        prefetch_factor=2 if num_workers > 0 else None,
    )

### 5) Run Pipeline and Quick Verification
Instantiate config, build pairs, create DataLoader, and run a short dry-run to validate input throughput.

In [ ]:
import json
from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm
import random

# =========================================================
# CONFIGURATION FOR PREPROCESSED DATASET
# =========================================================
PREPROCESSED_DATA_DIR = "/kaggle/input/datasets/vunhuduc/sketch-preprocessdata"

# =========================================================
# STEP 1: Load preprocessed metadata
# =========================================================
metadata_file = Path(PREPROCESSED_DATA_DIR) / "metadata.json"

print(f"Loading metadata from: {metadata_file}")
with open(metadata_file, "r", encoding="utf-8") as f:
    preprocessed_rows = json.load(f)

# STEP 2: Reconstruct absolute image paths for dataset
gan_rows = []
for row in preprocessed_rows:
    gan_rows.append({
        "category": row["category"],
        "filename_stem": row["filename_stem"],
        "real_path": str(Path(PREPROCESSED_DATA_DIR) / row["real_path"]),
        "sketch_path": str(Path(PREPROCESSED_DATA_DIR) / row["sketch_path"]),
        "sketch_method": row["sketch_method"]
    })

random.seed(42)
random.shuffle(gan_rows)
train_rows = gan_rows
fid_rows = gan_rows[:10000]

# Print pipeline summary
gan_summary = {
    "num_pairs": len(gan_rows),
    "train_pairs": len(train_rows),
    "fid_pairs": len(fid_rows),
    "sketch_method_counts": dict(Counter(r["sketch_method"] for r in gan_rows)),
}

print("\n" + "=" * 30)
print("PIPELINE SUMMARY (LOADED FROM PREPROCESSED DATA):")
print(f"- Total paired samples: {gan_summary['num_pairs']:,} (Train: {gan_summary['train_pairs']:,}, FID: {gan_summary['fid_pairs']:,})")
for method, count in gan_summary["sketch_method_counts"].items():
    print(f"  + {method}: {count:,} samples")
print("=" * 30)

# =========================================================
# STEP 3: Initialize DataLoader
# =========================================================
print("\nInitializing DataLoader...")
train_loader = build_gan_dataloader(
    rows=train_rows,
    batch_size=1,
    image_size=256,
    shuffle=True,
    apply_augmentation=True,
)

fid_loader = build_gan_dataloader(
    rows=fid_rows,
    batch_size=1,
    image_size=256,
    shuffle=False,
    apply_augmentation=False,
)
print(f"Train batches: {len(train_loader):,}, FID batches: {len(fid_loader):,}")

# =========================================================
# STEP 4: Dry run verification
# =========================================================
print("\nRunning dry-run for first 10 batches...")
for i, batch in enumerate(
    tqdm(
        train_loader,
        total=min(10, len(train_loader)),
        desc="Dry-run",
        unit="batch",
    )
):
    assert batch["sketch"].shape[1:] == (3, 256, 256)
    assert batch["real"].shape[1:] == (3, 256, 256)
    if i >= 9:
        break

print("\nPipeline is ready for training with sketch + z inputs.")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from pathlib import Path


def pil_to_uint8(image: Image.Image) -> np.ndarray:
    return np.asarray(image.convert("RGB"), dtype=np.uint8)


def tensor_to_uint8_img(tensor):
    """Convert a normalized CHW tensor in [-1, 1] to a displayable uint8 image."""
    image = tensor.detach().cpu().permute(1, 2, 0).numpy()
    image = ((image + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
    return image


def show_before_after_augment(sample_idx: int = 0):
    """Show images before and after augmentation for both sketch and real pairs."""
    if "gan_rows" not in globals() or len(gan_rows) == 0:
        print("gan_rows chưa có dữ liệu. Hãy chạy cell tạo DataLoader trước.")
        return

    if "SketchToRealGANDataset" not in globals():
        print("Dataset class chưa sẵn sàng. Hãy chạy cell Data Pipeline trước.")
        return

    sample = gan_rows[sample_idx]
    before_sketch = pil_to_uint8(Image.open(sample["sketch_path"]).convert("RGB"))
    before_real = pil_to_uint8(Image.open(sample["real_path"]).convert("RGB"))

    preview_dataset = SketchToRealGANDataset(
        rows=[sample],
        image_size=256,
        apply_augmentation=True,
    )
    aug_item = preview_dataset[0]
    after_sketch = tensor_to_uint8_img(aug_item["sketch"])
    after_real = tensor_to_uint8_img(aug_item["real"])

    fig, axes = plt.subplots(2, 2, figsize=(10, 10))

    axes[0, 0].imshow(before_sketch)
    axes[0, 0].set_title("Sketch before augment")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(after_sketch)
    axes[0, 1].set_title("Sketch after augment")
    axes[0, 1].axis("off")

    axes[1, 0].imshow(before_real)
    axes[1, 0].set_title("Real before augment")
    axes[1, 0].axis("off")

    axes[1, 1].imshow(after_real)
    axes[1, 1].set_title("Real after augment")
    axes[1, 1].axis("off")

    plt.suptitle(f"Before vs After Augmentation | sample_idx={sample_idx} | category={sample['category']}")
    plt.tight_layout()
    plt.show()


show_before_after_augment(sample_idx=0)

In [ ]:
# Cell 8
import matplotlib.pyplot as plt
import numpy as np
import random

def visualize_degradation_pipeline(dataset: SketchToRealGANDataset, num_samples: int = 3):
    """
    Trực quan hóa từng bước của quá trình Data Augmentation:
    Original -> Spatial Transform (cả 2) -> Degrade (chỉ Sketch)
    """
    if len(dataset) == 0:
        print("Dataset trống!")
        return

    # Lấy ngẫu nhiên các chỉ số
    indices = random.sample(range(len(dataset)), min(num_samples, len(dataset)))
    
    # Kích thước lưới: num_samples hàng, 5 cột (Real gốc, Sketch gốc, Real Spatial, Sketch Spatial, Sketch Degraded)
    fig, axes = plt.subplots(len(indices), 5, figsize=(18, 3.5 * len(indices)))
    
    # Xử lý trường hợp chỉ có 1 sample (axes sẽ là mảng 1D)
    if len(indices) == 1:
        axes = [axes]

    for i, idx in enumerate(indices):
        row = dataset.rows[idx]
        
        # 1. Load ảnh gốc (đã pad/resize)
        sketch_raw = dataset._load_img(row["sketch_path"])
        real_raw = dataset._load_img(row["real_path"])
        
        # 2. Áp dụng Spatial Augmentation (biến dạng không gian, áp dụng cho cả 2)
        transformed_spatial = dataset._spatial_augment(image=sketch_raw, real=real_raw)
        sketch_spatial = transformed_spatial["image"]
        real_spatial = transformed_spatial["real"]
        
        # 3. Áp dụng Sketch-only Augmentation (chỉ làm đứt nét/méo trên sketch)
        transformed_degrade = dataset._sketch_only_augment(image=sketch_spatial)
        sketch_degraded = transformed_degrade["image"]
        
        # --- Bắt đầu vẽ ---
        panels = [
            (real_raw, f"1. Real (Gốc)\n{row['category']}"),
            (sketch_raw, "2. Sketch (Gốc)"),
            (real_spatial, "3. Real (Sau Xoay/Crop)"),
            (sketch_spatial, "4. Sketch (Sau Xoay/Crop)\nChưa đứt nét"),
            (sketch_degraded, "5. Sketch (Degraded)\nĐã đứt nét & xô lệch"),
        ]
        
        for col, (img, title) in enumerate(panels):
            ax = axes[i][col]
            ax.imshow(img)
            ax.set_title(title, fontsize=11)
            ax.axis("off")
            
            # Thêm viền đỏ cho ảnh Degraded để dễ nhận biết
            if col == 4:
                for spine in ax.spines.values():
                    spine.set_edgecolor('red')
                    spine.set_linewidth(2)
                    spine.set_visible(True)

    plt.suptitle("Step-by-Step Data Augmentation & Sketch Degradation", fontsize=16, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.show()

# Lấy dataset từ dataloader đã tạo ở Cell 6 (hoặc khởi tạo mới)
# Đảm bảo bạn đã chạy Cell 5 (mới) và Cell 6 trước khi chạy code này
if 'gan_loader' in globals():
    my_dataset = gan_loader.dataset
    visualize_degradation_pipeline(my_dataset, num_samples=3)
else:
    print("Vui lòng khởi tạo 'gan_loader' trước.")

In [ ]:
import matplotlib.pyplot as plt
import random
from pathlib import Path


def preprocess_preview(path: str, target_size: int = 256):
    """Return raw, smart-padded, and resized versions of one image."""
    with Image.open(path) as src_img:
        raw = src_img.convert("RGB")

    w, h = raw.size
    canvas_side = max(w, h, target_size)
    pad_left = (canvas_side - w) // 2
    pad_top = (canvas_side - h) // 2
    pad_right = canvas_side - w - pad_left
    pad_bottom = canvas_side - h - pad_top

    padded = ImageOps.expand(
        raw,
        border=(pad_left, pad_top, pad_right, pad_bottom),
        fill=(255, 255, 255),
    )
    resized = padded.resize((target_size, target_size), Image.BICUBIC)
    return raw, padded, resized


def find_sketch_by_method(row: dict, method: str):
    """Find sketch image path for a specific method using pipeline config roots."""
    if "PIPE_CFG" not in globals():
        return None

    root = PIPE_CFG.sketch_roots.get(method)
    if root is None:
        return None

    category = row["category"]
    stem = row["filename_stem"]
    method_dir = Path(root) / category
    if not method_dir.exists():
        return None

    for ext in IMG_EXTENSIONS:
        candidate = method_dir / f"{stem}{ext}"
        if candidate.exists():
            return str(candidate)
    return None


def tensor_to_uint8_img(t: torch.Tensor) -> np.ndarray:
    """Convert CHW tensor in [-1, 1] to HWC uint8 image."""
    arr = t.detach().cpu().permute(1, 2, 0).numpy()
    arr = ((arr + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
    return arr


def visualize_full_samples(num_samples: int = 3, image_size: int = 256):
    """Visualize full sample information, including all sketch methods and augment result."""
    if "gan_rows" not in globals() or len(gan_rows) == 0:
        print("'gan_rows' chưa có dữ liệu. Hãy chạy Cell 15 trước.")
        return

    if "SketchToRealGANDataset" not in globals():
        print("Dataset class chưa sẵn sàng. Hãy chạy Cell 13 trước.")
        return

    sample_rows = random.sample(gan_rows, k=min(num_samples, len(gan_rows)))

    # Methods come from current config to guarantee complete per-method view.
    method_list = list(PIPE_CFG.sketch_roots.keys()) if "PIPE_CFG" in globals() else []

    # Build one augmented dataset instance to preview synchronized spatial augmentation.
    preview_dataset = SketchToRealGANDataset(
        rows=sample_rows,
        image_size=image_size,
        apply_augmentation=True,
    )

    # Columns: Real(raw/final), Selected Sketch(raw/final), each method final, augmented pair.
    n_cols = 4 + len(method_list) + 2
    fig, axes = plt.subplots(len(sample_rows), n_cols, figsize=(3.2 * n_cols, 3.0 * len(sample_rows)))
    if len(sample_rows) == 1:
        axes = [axes]

    print(f"Hiển thị {len(sample_rows)} mẫu | số cột: {n_cols}")

    for r, row in enumerate(sample_rows):
        # Real + selected sketch processing previews.
        real_raw, _, real_final = preprocess_preview(row["real_path"], target_size=image_size)
        sketch_raw, _, sketch_final = preprocess_preview(row["sketch_path"], target_size=image_size)

        # Augmented aligned pair preview.
        aug_item = preview_dataset[r]
        aug_sketch = tensor_to_uint8_img(aug_item["sketch"])
        aug_real = tensor_to_uint8_img(aug_item["real"])

        panels = [
            (real_raw, f"Real Raw\\n{Path(row['real_path']).name}"),
            (real_final, "Real Final"),
            (sketch_raw, f"Sketch Raw ({row['sketch_method']})"),
            (sketch_final, "Sketch Final (Selected)"),
        ]

        # Add each sketch type (hed/pencil/canny/...) for full comparison.
        for method in method_list:
            m_path = find_sketch_by_method(row, method)
            if m_path is None:
                blank = np.full((image_size, image_size, 3), 255, dtype=np.uint8)
                panels.append((blank, f"{method.upper()} (missing)"))
            else:
                _, _, m_final = preprocess_preview(m_path, target_size=image_size)
                panels.append((m_final, f"{method.upper()}"))

        panels.append((aug_sketch, "Aug Sketch"))
        panels.append((aug_real, "Aug Real"))

        for c, (img, title) in enumerate(panels):
            axes[r][c].imshow(img)
            axes[r][c].set_title(title, fontsize=9)
            axes[r][c].axis("off")

        # Write sample-level text at the first panel for quick traceability.
        axes[r][0].set_ylabel(
            f"Sample {r + 1}\\ncat: {row['category']}",
            rotation=90,
            fontsize=9,
            labelpad=10,
        )

    plt.suptitle(
        "Full Visualization: Real/Selected Sketch/All Sketch Methods/Augmented Pair",
        y=1.02,
        fontsize=14,
        fontweight="bold",
    )
    plt.tight_layout()
    plt.show()


# Run visualization
visualize_full_samples(num_samples=4, image_size=256)

In [ ]:
!pip install lpips

# pSp Architecture
Contains Encoders, StyleGAN2, and pSp wrapper.

In [ ]:
# ==========================================
# PSP ARCHITECTURE & STYLEGAN2 & ENCODERS
# (Inlined to run standalone in Kaggle)
# ==========================================
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Linear, Conv2d, BatchNorm1d, BatchNorm2d, PReLU, Dropout, Sequential, Module, ReLU, Sigmoid, MaxPool2d, AdaptiveAvgPool2d
from collections import namedtuple
from torch.autograd import Function

# --- Encoders Helpers ---
from collections import namedtuple
import torch
from torch.nn import Conv2d, BatchNorm2d, PReLU, ReLU, Sigmoid, MaxPool2d, AdaptiveAvgPool2d, Sequential, Module

"""
ArcFace implementation from [TreB1eN](https://github.com/TreB1eN/InsightFace_Pytorch)
"""


class Flatten(Module):
	def forward(self, input):
		return input.view(input.size(0), -1)


def l2_norm(input, axis=1):
	norm = torch.norm(input, 2, axis, True)
	output = torch.div(input, norm)
	return output


class Bottleneck(namedtuple('Block', ['in_channel', 'depth', 'stride'])):
	""" A named tuple describing a ResNet block. """


def get_block(in_channel, depth, num_units, stride=2):
	return [Bottleneck(in_channel, depth, stride)] + [Bottleneck(depth, depth, 1) for i in range(num_units - 1)]


def get_blocks(num_layers):
	if num_layers == 50:
		blocks = [
			get_block(in_channel=64, depth=64, num_units=3),
			get_block(in_channel=64, depth=128, num_units=4),
			get_block(in_channel=128, depth=256, num_units=14),
			get_block(in_channel=256, depth=512, num_units=3)
		]
	elif num_layers == 100:
		blocks = [
			get_block(in_channel=64, depth=64, num_units=3),
			get_block(in_channel=64, depth=128, num_units=13),
			get_block(in_channel=128, depth=256, num_units=30),
			get_block(in_channel=256, depth=512, num_units=3)
		]
	elif num_layers == 152:
		blocks = [
			get_block(in_channel=64, depth=64, num_units=3),
			get_block(in_channel=64, depth=128, num_units=8),
			get_block(in_channel=128, depth=256, num_units=36),
			get_block(in_channel=256, depth=512, num_units=3)
		]
	else:
		raise ValueError("Invalid number of layers: {}. Must be one of [50, 100, 152]".format(num_layers))
	return blocks


class SEModule(Module):
	def __init__(self, channels, reduction):
		super(SEModule, self).__init__()
		self.avg_pool = AdaptiveAvgPool2d(1)
		self.fc1 = Conv2d(channels, channels // reduction, kernel_size=1, padding=0, bias=False)
		self.relu = ReLU(inplace=True)
		self.fc2 = Conv2d(channels // reduction, channels, kernel_size=1, padding=0, bias=False)
		self.sigmoid = Sigmoid()

	def forward(self, x):
		module_input = x
		x = self.avg_pool(x)
		x = self.fc1(x)
		x = self.relu(x)
		x = self.fc2(x)
		x = self.sigmoid(x)
		return module_input * x


class bottleneck_IR(Module):
	def __init__(self, in_channel, depth, stride):
		super(bottleneck_IR, self).__init__()
		if in_channel == depth:
			self.shortcut_layer = MaxPool2d(1, stride)
		else:
			self.shortcut_layer = Sequential(
				Conv2d(in_channel, depth, (1, 1), stride, bias=False),
				BatchNorm2d(depth)
			)
		self.res_layer = Sequential(
			BatchNorm2d(in_channel),
			Conv2d(in_channel, depth, (3, 3), (1, 1), 1, bias=False), PReLU(depth),
			Conv2d(depth, depth, (3, 3), stride, 1, bias=False), BatchNorm2d(depth)
		)

	def forward(self, x):
		shortcut = self.shortcut_layer(x)
		res = self.res_layer(x)
		return res + shortcut


class bottleneck_IR_SE(Module):
	def __init__(self, in_channel, depth, stride):
		super(bottleneck_IR_SE, self).__init__()
		if in_channel == depth:
			self.shortcut_layer = MaxPool2d(1, stride)
		else:
			self.shortcut_layer = Sequential(
				Conv2d(in_channel, depth, (1, 1), stride, bias=False),
				BatchNorm2d(depth)
			)
		self.res_layer = Sequential(
			BatchNorm2d(in_channel),
			Conv2d(in_channel, depth, (3, 3), (1, 1), 1, bias=False),
			PReLU(depth),
			Conv2d(depth, depth, (3, 3), stride, 1, bias=False),
			BatchNorm2d(depth),
			SEModule(depth, 16)
		)

	def forward(self, x):
		shortcut = self.shortcut_layer(x)
		res = self.res_layer(x)
		return res + shortcut


# --- Model IRSE ---
from torch.nn import Linear, Conv2d, BatchNorm1d, BatchNorm2d, PReLU, Dropout, Sequential, Module

"""
Modified Backbone implementation from [TreB1eN](https://github.com/TreB1eN/InsightFace_Pytorch)
"""


class Backbone(Module):
	def __init__(self, input_size, num_layers, mode='ir', drop_ratio=0.4, affine=True):
		super(Backbone, self).__init__()
		assert input_size in [112, 224], "input_size should be 112 or 224"
		assert num_layers in [50, 100, 152], "num_layers should be 50, 100 or 152"
		assert mode in ['ir', 'ir_se'], "mode should be ir or ir_se"
		blocks = get_blocks(num_layers)
		if mode == 'ir':
			unit_module = bottleneck_IR
		elif mode == 'ir_se':
			unit_module = bottleneck_IR_SE
		self.input_layer = Sequential(Conv2d(3, 64, (3, 3), 1, 1, bias=False),
									  BatchNorm2d(64),
									  PReLU(64))
		if input_size == 112:
			self.output_layer = Sequential(BatchNorm2d(512),
			                               Dropout(drop_ratio),
			                               Flatten(),
			                               Linear(512 * 7 * 7, 512),
			                               BatchNorm1d(512, affine=affine))
		else:
			self.output_layer = Sequential(BatchNorm2d(512),
			                               Dropout(drop_ratio),
			                               Flatten(),
			                               Linear(512 * 14 * 14, 512),
			                               BatchNorm1d(512, affine=affine))

		modules = []
		for block in blocks:
			for bottleneck in block:
				modules.append(unit_module(bottleneck.in_channel,
										   bottleneck.depth,
										   bottleneck.stride))
		self.body = Sequential(*modules)

	def forward(self, x):
		x = self.input_layer(x)
		x = self.body(x)
		x = self.output_layer(x)
		return l2_norm(x)


def IR_50(input_size):
	"""Constructs a ir-50 model."""
	model = Backbone(input_size, num_layers=50, mode='ir', drop_ratio=0.4, affine=False)
	return model


def IR_101(input_size):
	"""Constructs a ir-101 model."""
	model = Backbone(input_size, num_layers=100, mode='ir', drop_ratio=0.4, affine=False)
	return model


def IR_152(input_size):
	"""Constructs a ir-152 model."""
	model = Backbone(input_size, num_layers=152, mode='ir', drop_ratio=0.4, affine=False)
	return model


def IR_SE_50(input_size):
	"""Constructs a ir_se-50 model."""
	model = Backbone(input_size, num_layers=50, mode='ir_se', drop_ratio=0.4, affine=False)
	return model


def IR_SE_101(input_size):
	"""Constructs a ir_se-101 model."""
	model = Backbone(input_size, num_layers=100, mode='ir_se', drop_ratio=0.4, affine=False)
	return model


def IR_SE_152(input_size):
	"""Constructs a ir_se-152 model."""
	model = Backbone(input_size, num_layers=152, mode='ir_se', drop_ratio=0.4, affine=False)
	return model


# --- fused_act (Native Fallback) ---
import os

import torch
from torch import nn
from torch.autograd import Function



class FusedLeakyReLU(nn.Module):
    def __init__(self, channel, negative_slope=0.2, scale=2 ** 0.5):
        super().__init__()

        self.bias = nn.Parameter(torch.zeros(channel))
        self.negative_slope = negative_slope
        self.scale = scale

    def forward(self, input):
        return fused_leaky_relu(input, self.bias, self.negative_slope, self.scale)


def fused_leaky_relu(input, bias, negative_slope=0.2, scale=2 ** 0.5):
    rest_dim = [1] * (input.ndim - bias.ndim - 1)
    return F.leaky_relu(input + bias.view(1, bias.shape[0], *rest_dim), negative_slope=negative_slope) * scale


# --- upfirdn2d (Native Fallback) ---
import os

import torch
from torch.autograd import Function



def upfirdn2d(input, kernel, up=1, down=1, pad=(0, 0)):
    out = upfirdn2d_native(input.permute(0, 2, 3, 1), kernel, up, up, down, down, pad[0], pad[1], pad[0], pad[1])
    return out.permute(0, 3, 1, 2)


def upfirdn2d_native(
        input, kernel, up_x, up_y, down_x, down_y, pad_x0, pad_x1, pad_y0, pad_y1
):
    _, in_h, in_w, minor = input.shape
    kernel_h, kernel_w = kernel.shape

    out = input.view(-1, in_h, 1, in_w, 1, minor)
    out = F.pad(out, [0, 0, 0, up_x - 1, 0, 0, 0, up_y - 1])
    out = out.view(-1, in_h * up_y, in_w * up_x, minor)

    out = F.pad(
        out, [0, 0, max(pad_x0, 0), max(pad_x1, 0), max(pad_y0, 0), max(pad_y1, 0)]
    )
    out = out[
          :,
          max(-pad_y0, 0): out.shape[1] - max(-pad_y1, 0),
          max(-pad_x0, 0): out.shape[2] - max(-pad_x1, 0),
          :,
          ]

    out = out.permute(0, 3, 1, 2)
    out = out.reshape(
        [-1, 1, in_h * up_y + pad_y0 + pad_y1, in_w * up_x + pad_x0 + pad_x1]
    )
    w = torch.flip(kernel, [0, 1]).view(1, 1, kernel_h, kernel_w)
    out = F.conv2d(out, w)
    out = out.reshape(
        -1,
        minor,
        in_h * up_y + pad_y0 + pad_y1 - kernel_h + 1,
        in_w * up_x + pad_x0 + pad_x1 - kernel_w + 1,
    )
    out = out.permute(0, 2, 3, 1)

    return out[:, ::down_y, ::down_x, :]


# --- StyleGAN2 Model ---
import math
import random
import torch
from torch import nn
from torch.nn import functional as F



class PixelNorm(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, input):
        return input * torch.rsqrt(torch.mean(input ** 2, dim=1, keepdim=True) + 1e-8)


def make_kernel(k):
    k = torch.tensor(k, dtype=torch.float32)

    if k.ndim == 1:
        k = k[None, :] * k[:, None]

    k /= k.sum()

    return k


class Upsample(nn.Module):
    def __init__(self, kernel, factor=2):
        super().__init__()

        self.factor = factor
        kernel = make_kernel(kernel) * (factor ** 2)
        self.register_buffer('kernel', kernel)

        p = kernel.shape[0] - factor

        pad0 = (p + 1) // 2 + factor - 1
        pad1 = p // 2

        self.pad = (pad0, pad1)

    def forward(self, input):
        out = upfirdn2d(input, self.kernel, up=self.factor, down=1, pad=self.pad)

        return out


class Downsample(nn.Module):
    def __init__(self, kernel, factor=2):
        super().__init__()

        self.factor = factor
        kernel = make_kernel(kernel)
        self.register_buffer('kernel', kernel)

        p = kernel.shape[0] - factor

        pad0 = (p + 1) // 2
        pad1 = p // 2

        self.pad = (pad0, pad1)

    def forward(self, input):
        out = upfirdn2d(input, self.kernel, up=1, down=self.factor, pad=self.pad)

        return out


class Blur(nn.Module):
    def __init__(self, kernel, pad, upsample_factor=1):
        super().__init__()

        kernel = make_kernel(kernel)

        if upsample_factor > 1:
            kernel = kernel * (upsample_factor ** 2)

        self.register_buffer('kernel', kernel)

        self.pad = pad

    def forward(self, input):
        out = upfirdn2d(input, self.kernel, pad=self.pad)

        return out


class EqualConv2d(nn.Module):
    def __init__(
            self, in_channel, out_channel, kernel_size, stride=1, padding=0, bias=True
    ):
        super().__init__()

        self.weight = nn.Parameter(
            torch.randn(out_channel, in_channel, kernel_size, kernel_size)
        )
        self.scale = 1 / math.sqrt(in_channel * kernel_size ** 2)

        self.stride = stride
        self.padding = padding

        if bias:
            self.bias = nn.Parameter(torch.zeros(out_channel))

        else:
            self.bias = None

    def forward(self, input):
        out = F.conv2d(
            input,
            self.weight * self.scale,
            bias=self.bias,
            stride=self.stride,
            padding=self.padding,
        )

        return out

    def __repr__(self):
        return (
            f'{self.__class__.__name__}({self.weight.shape[1]}, {self.weight.shape[0]},'
            f' {self.weight.shape[2]}, stride={self.stride}, padding={self.padding})'
        )


class EqualLinear(nn.Module):
    def __init__(
            self, in_dim, out_dim, bias=True, bias_init=0, lr_mul=1, activation=None
    ):
        super().__init__()

        self.weight = nn.Parameter(torch.randn(out_dim, in_dim).div_(lr_mul))

        if bias:
            self.bias = nn.Parameter(torch.zeros(out_dim).fill_(bias_init))

        else:
            self.bias = None

        self.activation = activation

        self.scale = (1 / math.sqrt(in_dim)) * lr_mul
        self.lr_mul = lr_mul

    def forward(self, input):
        if self.activation:
            out = F.linear(input, self.weight * self.scale)
            out = fused_leaky_relu(out, self.bias * self.lr_mul)

        else:
            out = F.linear(
                input, self.weight * self.scale, bias=self.bias * self.lr_mul
            )

        return out

    def __repr__(self):
        return (
            f'{self.__class__.__name__}({self.weight.shape[1]}, {self.weight.shape[0]})'
        )


class ScaledLeakyReLU(nn.Module):
    def __init__(self, negative_slope=0.2):
        super().__init__()

        self.negative_slope = negative_slope

    def forward(self, input):
        out = F.leaky_relu(input, negative_slope=self.negative_slope)

        return out * math.sqrt(2)


class ModulatedConv2d(nn.Module):
    def __init__(
            self,
            in_channel,
            out_channel,
            kernel_size,
            style_dim,
            demodulate=True,
            upsample=False,
            downsample=False,
            blur_kernel=[1, 3, 3, 1],
    ):
        super().__init__()

        self.eps = 1e-8
        self.kernel_size = kernel_size
        self.in_channel = in_channel
        self.out_channel = out_channel
        self.upsample = upsample
        self.downsample = downsample

        if upsample:
            factor = 2
            p = (len(blur_kernel) - factor) - (kernel_size - 1)
            pad0 = (p + 1) // 2 + factor - 1
            pad1 = p // 2 + 1

            self.blur = Blur(blur_kernel, pad=(pad0, pad1), upsample_factor=factor)

        if downsample:
            factor = 2
            p = (len(blur_kernel) - factor) + (kernel_size - 1)
            pad0 = (p + 1) // 2
            pad1 = p // 2

            self.blur = Blur(blur_kernel, pad=(pad0, pad1))

        fan_in = in_channel * kernel_size ** 2
        self.scale = 1 / math.sqrt(fan_in)
        self.padding = kernel_size // 2

        self.weight = nn.Parameter(
            torch.randn(1, out_channel, in_channel, kernel_size, kernel_size)
        )

        self.modulation = EqualLinear(style_dim, in_channel, bias_init=1)

        self.demodulate = demodulate

    def __repr__(self):
        return (
            f'{self.__class__.__name__}({self.in_channel}, {self.out_channel}, {self.kernel_size}, '
            f'upsample={self.upsample}, downsample={self.downsample})'
        )

    def forward(self, input, style):
        batch, in_channel, height, width = input.shape

        style = self.modulation(style).view(batch, 1, in_channel, 1, 1)
        weight = self.scale * self.weight * style

        if self.demodulate:
            demod = torch.rsqrt(weight.pow(2).sum([2, 3, 4]) + 1e-8)
            weight = weight * demod.view(batch, self.out_channel, 1, 1, 1)

        weight = weight.view(
            batch * self.out_channel, in_channel, self.kernel_size, self.kernel_size
        )

        if self.upsample:
            input = input.view(1, batch * in_channel, height, width)
            weight = weight.view(
                batch, self.out_channel, in_channel, self.kernel_size, self.kernel_size
            )
            weight = weight.transpose(1, 2).reshape(
                batch * in_channel, self.out_channel, self.kernel_size, self.kernel_size
            )
            out = F.conv_transpose2d(input, weight, padding=0, stride=2, groups=batch)
            _, _, height, width = out.shape
            out = out.view(batch, self.out_channel, height, width)
            out = self.blur(out)

        elif self.downsample:
            input = self.blur(input)
            _, _, height, width = input.shape
            input = input.view(1, batch * in_channel, height, width)
            out = F.conv2d(input, weight, padding=0, stride=2, groups=batch)
            _, _, height, width = out.shape
            out = out.view(batch, self.out_channel, height, width)

        else:
            input = input.view(1, batch * in_channel, height, width)
            out = F.conv2d(input, weight, padding=self.padding, groups=batch)
            _, _, height, width = out.shape
            out = out.view(batch, self.out_channel, height, width)

        return out


class NoiseInjection(nn.Module):
    def __init__(self):
        super().__init__()

        self.weight = nn.Parameter(torch.zeros(1))

    def forward(self, image, noise=None):
        if noise is None:
            batch, _, height, width = image.shape
            noise = image.new_empty(batch, 1, height, width).normal_()

        return image + self.weight * noise


class ConstantInput(nn.Module):
    def __init__(self, channel, size=4):
        super().__init__()

        self.input = nn.Parameter(torch.randn(1, channel, size, size))

    def forward(self, input):
        batch = input.shape[0]
        out = self.input.repeat(batch, 1, 1, 1)

        return out


class StyledConv(nn.Module):
    def __init__(
            self,
            in_channel,
            out_channel,
            kernel_size,
            style_dim,
            upsample=False,
            blur_kernel=[1, 3, 3, 1],
            demodulate=True,
    ):
        super().__init__()

        self.conv = ModulatedConv2d(
            in_channel,
            out_channel,
            kernel_size,
            style_dim,
            upsample=upsample,
            blur_kernel=blur_kernel,
            demodulate=demodulate,
        )

        self.noise = NoiseInjection()
        # self.bias = nn.Parameter(torch.zeros(1, out_channel, 1, 1))
        # self.activate = ScaledLeakyReLU(0.2)
        self.activate = FusedLeakyReLU(out_channel)

    def forward(self, input, style, noise=None):
        out = self.conv(input, style)
        out = self.noise(out, noise=noise)
        # out = out + self.bias
        out = self.activate(out)

        return out


class ToRGB(nn.Module):
    def __init__(self, in_channel, style_dim, upsample=True, blur_kernel=[1, 3, 3, 1]):
        super().__init__()

        if upsample:
            self.upsample = Upsample(blur_kernel)

        self.conv = ModulatedConv2d(in_channel, 3, 1, style_dim, demodulate=False)
        self.bias = nn.Parameter(torch.zeros(1, 3, 1, 1))

    def forward(self, input, style, skip=None):
        out = self.conv(input, style)
        out = out + self.bias

        if skip is not None:
            skip = self.upsample(skip)

            out = out + skip

        return out


class Generator(nn.Module):
    def __init__(
            self,
            size,
            style_dim,
            n_mlp,
            channel_multiplier=2,
            blur_kernel=[1, 3, 3, 1],
            lr_mlp=0.01,
    ):
        super().__init__()

        self.size = size

        self.style_dim = style_dim

        layers = [PixelNorm()]

        for i in range(n_mlp):
            layers.append(
                EqualLinear(
                    style_dim, style_dim, lr_mul=lr_mlp, activation='fused_lrelu'
                )
            )

        self.style = nn.Sequential(*layers)

        self.channels = {
            4: 512,
            8: 512,
            16: 512,
            32: 512,
            64: 256 * channel_multiplier,
            128: 128 * channel_multiplier,
            256: 64 * channel_multiplier,
            512: 32 * channel_multiplier,
            1024: 16 * channel_multiplier,
        }

        self.input = ConstantInput(self.channels[4])
        self.conv1 = StyledConv(
            self.channels[4], self.channels[4], 3, style_dim, blur_kernel=blur_kernel
        )
        self.to_rgb1 = ToRGB(self.channels[4], style_dim, upsample=False)

        self.log_size = int(math.log(size, 2))
        self.num_layers = (self.log_size - 2) * 2 + 1

        self.convs = nn.ModuleList()
        self.upsamples = nn.ModuleList()
        self.to_rgbs = nn.ModuleList()
        self.noises = nn.Module()

        in_channel = self.channels[4]

        for layer_idx in range(self.num_layers):
            res = (layer_idx + 5) // 2
            shape = [1, 1, 2 ** res, 2 ** res]
            self.noises.register_buffer(f'noise_{layer_idx}', torch.randn(*shape))

        for i in range(3, self.log_size + 1):
            out_channel = self.channels[2 ** i]

            self.convs.append(
                StyledConv(
                    in_channel,
                    out_channel,
                    3,
                    style_dim,
                    upsample=True,
                    blur_kernel=blur_kernel,
                )
            )

            self.convs.append(
                StyledConv(
                    out_channel, out_channel, 3, style_dim, blur_kernel=blur_kernel
                )
            )

            self.to_rgbs.append(ToRGB(out_channel, style_dim))

            in_channel = out_channel

        self.n_latent = self.log_size * 2 - 2

    def make_noise(self):
        device = self.input.input.device

        noises = [torch.randn(1, 1, 2 ** 2, 2 ** 2, device=device)]

        for i in range(3, self.log_size + 1):
            for _ in range(2):
                noises.append(torch.randn(1, 1, 2 ** i, 2 ** i, device=device))

        return noises

    def mean_latent(self, n_latent):
        latent_in = torch.randn(
            n_latent, self.style_dim, device=self.input.input.device
        )
        latent = self.style(latent_in).mean(0, keepdim=True)

        return latent

    def get_latent(self, input):
        return self.style(input)

    def forward(
            self,
            styles,
            return_latents=False,
            return_features=False,
            inject_index=None,
            truncation=1,
            truncation_latent=None,
            input_is_latent=False,
            noise=None,
            randomize_noise=True,
    ):
        if not input_is_latent:
            styles = [self.style(s) for s in styles]

        if noise is None:
            if randomize_noise:
                noise = [None] * self.num_layers
            else:
                noise = [
                    getattr(self.noises, f'noise_{i}') for i in range(self.num_layers)
                ]

        if truncation < 1:
            style_t = []

            for style in styles:
                style_t.append(
                    truncation_latent + truncation * (style - truncation_latent)
                )

            styles = style_t

        if len(styles) < 2:
            inject_index = self.n_latent

            if styles[0].ndim < 3:
                latent = styles[0].unsqueeze(1).repeat(1, inject_index, 1)
            else:
                latent = styles[0]

        else:
            if inject_index is None:
                inject_index = random.randint(1, self.n_latent - 1)

            latent = styles[0].unsqueeze(1).repeat(1, inject_index, 1)
            latent2 = styles[1].unsqueeze(1).repeat(1, self.n_latent - inject_index, 1)

            latent = torch.cat([latent, latent2], 1)

        out = self.input(latent)
        out = self.conv1(out, latent[:, 0], noise=noise[0])

        skip = self.to_rgb1(out, latent[:, 1])

        i = 1
        for conv1, conv2, noise1, noise2, to_rgb in zip(
                self.convs[::2], self.convs[1::2], noise[1::2], noise[2::2], self.to_rgbs
        ):
            out = conv1(out, latent[:, i], noise=noise1)
            out = conv2(out, latent[:, i + 1], noise=noise2)
            skip = to_rgb(out, latent[:, i + 2], skip)

            i += 2

        image = skip

        if return_latents:
            return image, latent
        elif return_features:
            return image, out
        else:
            return image, None


class ConvLayer(nn.Sequential):
    def __init__(
            self,
            in_channel,
            out_channel,
            kernel_size,
            downsample=False,
            blur_kernel=[1, 3, 3, 1],
            bias=True,
            activate=True,
    ):
        layers = []

        if downsample:
            factor = 2
            p = (len(blur_kernel) - factor) + (kernel_size - 1)
            pad0 = (p + 1) // 2
            pad1 = p // 2

            layers.append(Blur(blur_kernel, pad=(pad0, pad1)))

            stride = 2
            self.padding = 0

        else:
            stride = 1
            self.padding = kernel_size // 2

        layers.append(
            EqualConv2d(
                in_channel,
                out_channel,
                kernel_size,
                padding=self.padding,
                stride=stride,
                bias=bias and not activate,
            )
        )

        if activate:
            if bias:
                layers.append(FusedLeakyReLU(out_channel))

            else:
                layers.append(ScaledLeakyReLU(0.2))

        super().__init__(*layers)


class ResBlock(nn.Module):
    def __init__(self, in_channel, out_channel, blur_kernel=[1, 3, 3, 1]):
        super().__init__()

        self.conv1 = ConvLayer(in_channel, in_channel, 3)
        self.conv2 = ConvLayer(in_channel, out_channel, 3, downsample=True)

        self.skip = ConvLayer(
            in_channel, out_channel, 1, downsample=True, activate=False, bias=False
        )

    def forward(self, input):
        out = self.conv1(input)
        out = self.conv2(out)

        skip = self.skip(input)
        out = (out + skip) / math.sqrt(2)

        return out


class Discriminator(nn.Module):
    def __init__(self, size, channel_multiplier=2, blur_kernel=[1, 3, 3, 1]):
        super().__init__()

        channels = {
            4: 512,
            8: 512,
            16: 512,
            32: 512,
            64: 256 * channel_multiplier,
            128: 128 * channel_multiplier,
            256: 64 * channel_multiplier,
            512: 32 * channel_multiplier,
            1024: 16 * channel_multiplier,
        }

        convs = [ConvLayer(3, channels[size], 1)]

        log_size = int(math.log(size, 2))

        in_channel = channels[size]

        for i in range(log_size, 2, -1):
            out_channel = channels[2 ** (i - 1)]

            convs.append(ResBlock(in_channel, out_channel, blur_kernel))

            in_channel = out_channel

        self.convs = nn.Sequential(*convs)

        self.stddev_group = 4
        self.stddev_feat = 1

        self.final_conv = ConvLayer(in_channel + 1, channels[4], 3)
        self.final_linear = nn.Sequential(
            EqualLinear(channels[4] * 4 * 4, channels[4], activation='fused_lrelu'),
            EqualLinear(channels[4], 1),
        )

    def forward(self, input):
        out = self.convs(input)

        batch, channel, height, width = out.shape
        group = min(batch, self.stddev_group)
        stddev = out.view(
            group, -1, self.stddev_feat, channel // self.stddev_feat, height, width
        )
        stddev = torch.sqrt(stddev.var(0, unbiased=False) + 1e-8)
        stddev = stddev.mean([2, 3, 4], keepdims=True).squeeze(2)
        stddev = stddev.repeat(group, 1, height, width)
        out = torch.cat([out, stddev], 1)

        out = self.final_conv(out)

        out = out.view(batch, -1)
        out = self.final_linear(out)

        return out


# --- PSP Encoders ---
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.nn import Linear, Conv2d, BatchNorm2d, PReLU, Sequential, Module



class GradualStyleBlock(Module):
    def __init__(self, in_c, out_c, spatial):
        super(GradualStyleBlock, self).__init__()
        self.out_c = out_c
        self.spatial = spatial
        num_pools = int(np.log2(spatial))
        modules = []
        modules += [Conv2d(in_c, out_c, kernel_size=3, stride=2, padding=1),
                    nn.LeakyReLU()]
        for i in range(num_pools - 1):
            modules += [
                Conv2d(out_c, out_c, kernel_size=3, stride=2, padding=1),
                nn.LeakyReLU()
            ]
        self.convs = nn.Sequential(*modules)
        self.linear = EqualLinear(out_c, out_c, lr_mul=1)

    def forward(self, x):
        x = self.convs(x)
        x = x.view(-1, self.out_c)
        x = self.linear(x)
        return x


import torchvision.models as models

class GradualStyleEncoder(Module):
    def __init__(self, num_layers=50, mode='ir', opts=None):
        super(GradualStyleEncoder, self).__init__()
        print("Using ResNet50 (ImageNet) for GradualStyleEncoder")
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        
        if opts.input_nc != 3:
            resnet.conv1 = nn.Conv2d(opts.input_nc, 64, kernel_size=7, stride=2, padding=3, bias=False)
            nn.init.kaiming_normal_(resnet.conv1.weight, mode='fan_out', nonlinearity='relu')
            
        self.input_layer = Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool
        )
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        
        self.styles = nn.ModuleList()
        self.style_count = opts.n_styles
        self.coarse_ind = 3
        self.middle_ind = 7
        for i in range(self.style_count):
            if i < self.coarse_ind:
                style = GradualStyleBlock(512, 512, 16)
            elif i < self.middle_ind:
                style = GradualStyleBlock(512, 512, 32)
            else:
                style = GradualStyleBlock(512, 512, 64)
            self.styles.append(style)
            
        self.latlayer0 = nn.Conv2d(1024, 512, kernel_size=1, stride=1, padding=0)
        self.latlayer1 = nn.Conv2d(512, 512, kernel_size=1, stride=1, padding=0)
        self.latlayer2 = nn.Conv2d(256, 512, kernel_size=1, stride=1, padding=0)

    def _upsample_add(self, x, y):
        '''Upsample and add two feature maps.'''
        _, _, H, W = y.size()
        return F.interpolate(x, size=(H, W), mode='bilinear', align_corners=True) + y

    def forward(self, x):
        x = self.input_layer(x)
        c1 = self.layer1(x)
        c2 = self.layer2(c1)
        c3_raw = self.layer3(c2)
        
        c3 = self.latlayer0(c3_raw)
        
        latents = []
        for j in range(self.coarse_ind):
            latents.append(self.styles[j](c3))

        p2 = self._upsample_add(c3, self.latlayer1(c2))
        for j in range(self.coarse_ind, self.middle_ind):
            latents.append(self.styles[j](p2))

        p1 = self._upsample_add(p2, self.latlayer2(c1))
        for j in range(self.middle_ind, self.style_count):
            latents.append(self.styles[j](p1))

        out = torch.stack(latents, dim=1)
        return out


class BackboneEncoderUsingLastLayerIntoW(Module):
    def __init__(self, num_layers, mode='ir', opts=None):
        super(BackboneEncoderUsingLastLayerIntoW, self).__init__()
        print('Using BackboneEncoderUsingLastLayerIntoW')
        assert num_layers in [50, 100, 152], 'num_layers should be 50,100, or 152'
        assert mode in ['ir', 'ir_se'], 'mode should be ir or ir_se'
        blocks = get_blocks(num_layers)
        if mode == 'ir':
            unit_module = bottleneck_IR
        elif mode == 'ir_se':
            unit_module = bottleneck_IR_SE
        self.input_layer = Sequential(Conv2d(opts.input_nc, 64, (3, 3), 1, 1, bias=False),
                                      BatchNorm2d(64),
                                      PReLU(64))
        self.output_pool = torch.nn.AdaptiveAvgPool2d((1, 1))
        self.linear = EqualLinear(512, 512, lr_mul=1)
        modules = []
        for block in blocks:
            for bottleneck in block:
                modules.append(unit_module(bottleneck.in_channel,
                                           bottleneck.depth,
                                           bottleneck.stride))
        self.body = Sequential(*modules)

    def forward(self, x):
        x = self.input_layer(x)
        x = self.body(x)
        x = self.output_pool(x)
        x = x.view(-1, 512)
        x = self.linear(x)
        return x


class BackboneEncoderUsingLastLayerIntoWPlus(Module):
    def __init__(self, num_layers, mode='ir', opts=None):
        super(BackboneEncoderUsingLastLayerIntoWPlus, self).__init__()
        print('Using BackboneEncoderUsingLastLayerIntoWPlus')
        assert num_layers in [50, 100, 152], 'num_layers should be 50,100, or 152'
        assert mode in ['ir', 'ir_se'], 'mode should be ir or ir_se'
        blocks = get_blocks(num_layers)
        if mode == 'ir':
            unit_module = bottleneck_IR
        elif mode == 'ir_se':
            unit_module = bottleneck_IR_SE
        self.n_styles = opts.n_styles
        self.input_layer = Sequential(Conv2d(opts.input_nc, 64, (3, 3), 1, 1, bias=False),
                                      BatchNorm2d(64),
                                      PReLU(64))
        self.output_layer_2 = Sequential(BatchNorm2d(512),
                                         torch.nn.AdaptiveAvgPool2d((7, 7)),
                                         Flatten(),
                                         Linear(512 * 7 * 7, 512))
        self.linear = EqualLinear(512, 512 * self.n_styles, lr_mul=1)
        modules = []
        for block in blocks:
            for bottleneck in block:
                modules.append(unit_module(bottleneck.in_channel,
                                           bottleneck.depth,
                                           bottleneck.stride))
        self.body = Sequential(*modules)

    def forward(self, x):
        x = self.input_layer(x)
        x = self.body(x)
        x = self.output_layer_2(x)
        x = self.linear(x)
        x = x.view(-1, self.n_styles, 512)
        return x


# --- pSp Main Architecture ---
"""
This file defines the core research contribution
"""
import matplotlib
matplotlib.use('Agg')
import math

import torch
from torch import nn


def get_keys(d, name):
	if 'state_dict' in d:
		d = d['state_dict']
	d_filt = {k[len(name) + 1:]: v for k, v in d.items() if k[:len(name)] == name}
	return d_filt


class pSp(nn.Module):

	def __init__(self, opts):
		super(pSp, self).__init__()
		self.set_opts(opts)
		# compute number of style inputs based on the output resolution
		self.opts.n_styles = int(math.log(self.opts.output_size, 2)) * 2 - 2
		# Define architecture
		self.encoder = self.set_encoder()
		self.decoder = Generator(self.opts.output_size, 512, 8)
		self.face_pool = torch.nn.AdaptiveAvgPool2d((256, 256))
		# Load weights if needed
		self.load_weights()

	def set_encoder(self):
		if self.opts.encoder_type == 'GradualStyleEncoder':
			encoder = GradualStyleEncoder(50, 'ir_se', self.opts)
		elif self.opts.encoder_type == 'BackboneEncoderUsingLastLayerIntoW':
			encoder = BackboneEncoderUsingLastLayerIntoW(50, 'ir_se', self.opts)
		elif self.opts.encoder_type == 'BackboneEncoderUsingLastLayerIntoWPlus':
			encoder = BackboneEncoderUsingLastLayerIntoWPlus(50, 'ir_se', self.opts)
		else:
			raise Exception('{} is not a valid encoders'.format(self.opts.encoder_type))
		return encoder

	def load_weights(self):
		if getattr(self.opts, 'checkpoint_path', None) is not None:
			print('Loading pSp from checkpoint: {}'.format(self.opts.checkpoint_path))
			ckpt = torch.load(self.opts.checkpoint_path, map_location='cpu')
			def load_safe(module, state_dict):
				model_dict = module.state_dict()
				pretrained_dict = {k: v for k, v in state_dict.items() if k in model_dict and v.shape == model_dict[k].shape}
				model_dict.update(pretrained_dict)
				module.load_state_dict(model_dict)
			load_safe(self.encoder, get_keys(ckpt, 'encoder'))
			load_safe(self.decoder, get_keys(ckpt, 'decoder'))
			self.__load_latent_avg(ckpt)
		else:
			if getattr(self.opts, 'stylegan_weights', None) is not None:
				# Encoder now uses ResNet50 (ImageNet), skip ir_se50 loading
				print('Encoder is using PyTorch ResNet50 ImageNet pretrained weights.')
				
				print('Loading decoder weights from pretrained!')
				ckpt = torch.load(self.opts.stylegan_weights)
				self.decoder.load_state_dict(ckpt['g_ema'], strict=False)
				if self.opts.learn_in_w:
					self.__load_latent_avg(ckpt, repeat=1)
				else:
					self.__load_latent_avg(ckpt, repeat=self.opts.n_styles)
			else:
				print('No checkpoint or stylegan_weights provided. Training completely from scratch!')
				self.latent_avg = None

	def forward(self, x, resize=True, latent_mask=None, input_code=False, randomize_noise=True,
	            inject_latent=None, return_latents=False, alpha=None):
		if input_code:
			codes = x
		else:
			codes = self.encoder(x)
			# normalize with respect to the center of an average face
			if self.opts.start_from_latent_avg and self.latent_avg is not None:
				if self.opts.learn_in_w:
					codes = codes + self.latent_avg.repeat(codes.shape[0], 1)
				else:
					codes = codes + self.latent_avg.repeat(codes.shape[0], 1, 1)


		if latent_mask is not None:
			for i in latent_mask:
				if inject_latent is not None:
					if alpha is not None:
						codes[:, i] = alpha * inject_latent[:, i] + (1 - alpha) * codes[:, i]
					else:
						codes[:, i] = inject_latent[:, i]
				else:
					codes[:, i] = 0

		input_is_latent = not input_code
		images, result_latent = self.decoder([codes],
		                                     input_is_latent=input_is_latent,
		                                     randomize_noise=randomize_noise,
		                                     return_latents=return_latents)

		if resize:
			images = self.face_pool(images)

		if return_latents:
			return images, result_latent
		else:
			return images

	def set_opts(self, opts):
		self.opts = opts

	def __load_latent_avg(self, ckpt, repeat=None):
		if 'latent_avg' in ckpt:
			self.latent_avg = ckpt['latent_avg'].to(self.opts.device)
			if self.latent_avg.dim() == 2 and self.latent_avg.shape[0] != self.opts.n_styles:
				import torch
				if self.latent_avg.shape[0] > self.opts.n_styles:
					self.latent_avg = self.latent_avg[:self.opts.n_styles, :]
				else:
					diff = self.opts.n_styles - self.latent_avg.shape[0]
					self.latent_avg = torch.cat([self.latent_avg, self.latent_avg[-1:].repeat(diff, 1)], dim=0)
			if repeat is not None:
				self.latent_avg = self.latent_avg.repeat(repeat, 1)
		else:
			self.latent_avg = None


## Training Configuration and Utilities

In [ ]:
import csv
import json
import time
import random
from pathlib import Path
from dataclasses import dataclass, asdict
import torch
from torch.optim import Adam
from torch.amp import GradScaler, autocast
import itertools
import matplotlib.pyplot as plt
import torchvision.transforms as transforms

@dataclass
class TrainConfig:
    # Training
    epochs: int = 3
    batch_size: int = 1
    lr: float = 0.0001
    weight_decay: float = 0.0001
    
    # Model config
    output_size: int = 256
    encoder_type: str = 'GradualStyleEncoder'
    n_styles: int = 14 # log2(256)*2 - 2
    input_nc: int = 3
    label_nc: int = 0
    start_from_latent_avg: bool = True
    learn_in_w: bool = False
    checkpoint_path: str = None
    stylegan_weights: str = None # Path to StyleGAN2 pretrained if not using checkpoint
    
    # Loss weights
    lambda_l2: float = 1.0
    lambda_lpips: float = 0.8
    # ID loss is omitted by default for Sketch to Clothing.
    
    # Logging
    wandb_key: str = "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"
    log_every_steps: int = 100
    val_every_steps: int = 10000
    save_every: int = 1
    sample_every: int = 1
    
    # Mixed precision
    use_amp: bool = True
    
    # Resuming
    resume_checkpoint: str = "/kaggle/input/models/vunhuduc/pspv2-eps1/pytorch/default/1/epoch_1.pt"
    resume_strict: bool = True
    device: str = "cuda"

def get_output_root(run_name: str = None) -> Path:
    base = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./outputs')
    if run_name:
        base = base / run_name
    
    (base / 'checkpoints').mkdir(parents=True, exist_ok=True)
    (base / 'samples').mkdir(parents=True, exist_ok=True)
    return base

def save_sample_grid(real_A, real_B, fake_B, out_path):
    def denorm(t):
        return ((t + 1) * 127.5).clamp(0, 255).byte().cpu().permute(1, 2, 0).numpy()
    
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    
    axes[0].set_title('Sketch Input')
    axes[0].imshow(denorm(real_A[0]))
    axes[0].axis('off')
    
    axes[1].set_title('Generated Image')
    axes[1].imshow(denorm(fake_B[0]))
    axes[1].axis('off')
    
    axes[2].set_title('Target Real')
    axes[2].imshow(denorm(real_B[0]))
    axes[2].axis('off')
    
    plt.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

def save_checkpoint(out_dir, epoch, net, opt, scaler, config, filename='last.pt'):
    ckpt = {
        'epoch': epoch,
        'state_dict': net.state_dict(),
        'opt_state_dict': opt.state_dict(),
        'scaler_state_dict': scaler.state_dict() if scaler is not None else None,
        'config': asdict(config),
    }
    torch.save(ckpt, out_dir / 'checkpoints' / filename)

## Training Loop

In [ ]:
import wandb
import lpips
from torchmetrics.image.fid import FrechetInceptionDistance

def train_psp(net, train_loader, fid_loader, config, out_dir):
    if config.wandb_key:
        wandb.login(key=config.wandb_key)
    run_name = f"psp_{config.epochs}"
    wandb.init(project="sketch-psp", name=run_name, config=asdict(config))
    
    fid_metric = FrechetInceptionDistance(feature=2048, normalize=True).to(DEVICE)
    criterion_l2 = nn.MSELoss().to(DEVICE)
    criterion_lpips = lpips.LPIPS(net='alex').to(DEVICE)

    for param in net.decoder.parameters():
        param.requires_grad = False

    opt = Adam(net.encoder.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    scaler = GradScaler('cuda', enabled=(config.use_amp and DEVICE.type == 'cuda'))

    start_epoch = 1
    if config.resume_checkpoint and Path(config.resume_checkpoint).exists():
        print(f"Loading checkpoint from {config.resume_checkpoint}")
        ckpt = torch.load(config.resume_checkpoint, map_location='cpu')
        net.load_state_dict(ckpt['state_dict'], strict=config.resume_strict)
        if 'opt_state_dict' in ckpt: opt.load_state_dict(ckpt['opt_state_dict'])
        if 'scaler_state_dict' in ckpt and ckpt['scaler_state_dict'] is not None:
            scaler.load_state_dict(ckpt['scaler_state_dict'])
        start_epoch = ckpt.get('epoch', 0) + 1
        print(f"Successfully resumed from epoch {start_epoch - 1}")

    global_step = (start_epoch - 1) * len(train_loader)
    best_fid = float('inf')
    
    for epoch in range(start_epoch, config.epochs + 1):
        net.train()

        for batch_idx, batch in enumerate(train_loader):
            sketch = batch['sketch'].to(DEVICE, non_blocking=True)
            real = batch['real'].to(DEVICE, non_blocking=True)
            
            opt.zero_grad(set_to_none=True)
            
            with autocast(device_type='cuda', enabled=scaler.is_enabled()):
                fake, latents = net(sketch, return_latents=True)
                
                loss_l2 = criterion_l2(fake, real)
                loss_lpips = criterion_lpips(fake, real).mean()
                
                loss = loss_l2 * config.lambda_l2 + loss_lpips * config.lambda_lpips

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.encoder.parameters(), 5.0)
            scaler.step(opt)
            scaler.update()
            
            global_step += 1

            if global_step % config.log_every_steps == 0:
                print(
                    f"Epoch {epoch}/{config.epochs} | "
                    f"Batch {batch_idx + 1}/{len(train_loader)} | "
                    f"Step {global_step} | "
                    f"L2: {loss_l2.item():.4f} | "
                    f"LPIPS: {loss_lpips.item():.4f} | "
                    f"Total: {loss.item():.4f}"
                )
                try:
                    wandb.log({
                        "train/loss_l2": loss_l2.item(),
                        "train/loss_lpips": loss_lpips.item(),
                        "train/loss_total": loss.item(),
                        "global_step": global_step
                    })
                except Exception as e:
                    print("Failed to log step to wandb:", e)
                    
            if global_step % config.val_every_steps == 0:
                net.eval()
                
                fid_metric.reset()
                
                print(f"\nRunning FID calculation at step {global_step}...")
                with torch.no_grad():
                    for fid_batch in fid_loader:
                        fid_sketch = fid_batch['sketch'].to(DEVICE, non_blocking=True)
                        fid_real = fid_batch['real'].to(DEVICE, non_blocking=True)
                        
                        with autocast(device_type='cuda', enabled=scaler.is_enabled()):
                            fid_fake = net(fid_sketch)

                        fid_fake_norm = (fid_fake + 1) / 2
                        fid_real_norm = (fid_real + 1) / 2
                        
                        fid_metric.update(fid_real_norm, real=True)
                        fid_metric.update(fid_fake_norm, real=False)

                fid_score = fid_metric.compute().item()
                
                print(f"Step {global_step} | Train FID: {fid_score:.4f}")
                
                try:
                    wandb.log({
                        "train/fid": fid_score,
                        "global_step": global_step,
                        "epoch": epoch
                    })
                except Exception as e:
                    print("Failed to log FID to wandb:", e)

                if fid_score < best_fid:
                    best_fid = fid_score
                    save_checkpoint(out_dir, epoch, net, opt, scaler, config, filename='best_model.pt')
                    print(f"New best model saved at step {global_step} with FID: {best_fid:.4f}")
                
                net.train()

        if epoch % config.save_every == 0:
            save_checkpoint(out_dir, epoch, net, opt, scaler, config, filename='last.pt')
            save_checkpoint(out_dir, epoch, net, opt, scaler, config, filename=f'epoch_{epoch}.pt')

        if epoch % config.sample_every == 0:
            sample_path = out_dir / 'samples' / f'epoch_{epoch}.png'
            with torch.no_grad():
                net.eval()
                val_sample_batch = next(iter(fid_loader))
                sample_A = val_sample_batch['sketch'][:1].to(DEVICE)
                sample_B = val_sample_batch['real'][:1].to(DEVICE)
                
                fake_B_val = net(sample_A)
                save_sample_grid(sample_A, sample_B, fake_B_val, sample_path)
        
        torch.cuda.empty_cache()

## Execution

In [ ]:
# Define paths for pretrained backbones if not using a full pSp checkpoint
# model_paths = {
#     'ir_se50': '/kaggle/input/your-pretrained-model-path/model_ir_se50.pth'
# }

if __name__ == '__main__':
    # Initialize Configuration
    config = TrainConfig()
    
    # ---------------------------------------------------------
    # PRETRAINED MODEL CONFIGURATION
    # ---------------------------------------------------------
    # Path to the pre-trained pSp weights (contains both encoder and decoder)
    # Set to None if you want to train from scratch or use separate weights.
    config.checkpoint_path = None  # Train completely from scratch (set to None)
    
    # If using separate weights (checkpoint_path = None), set the StyleGAN2 weights path here:
    #config.stylegan_weights = '/kaggle/input/your-pretrained-model-path/stylegan2_weights.pt'
    # ---------------------------------------------------------

    # Initialize Output Directory
    out_dir = get_output_root(run_name='psp_training')
    print(f"Outputs will be saved to: {out_dir}")

    # Build DataLoader
    train_loader = build_gan_dataloader(
        rows=train_rows,
        batch_size=config.batch_size,
        image_size=config.output_size,
        shuffle=True,
        apply_augmentation=True,
    )
    fid_loader = build_gan_dataloader(
        rows=fid_rows,
        batch_size=config.batch_size,
        image_size=config.output_size,
        shuffle=False,
        apply_augmentation=False,
    )

    # Initialize pSp Network
    print("Initializing pSp architecture...")
    net = pSp(config).to(DEVICE)

    # Start Training
    print("Starting pSp training...")
    train_psp(net, train_loader, fid_loader, config, out_dir)